<a href="https://colab.research.google.com/github/danielle-arvidson/duty-manager-rag/blob/main/duty_manager_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duty manager decision recall: RAG build and evaluation

Retrieval-augmented generation over 13 mock duty manager records. Compares two embedding models, adds a reranker, and compares Claude Haiku against Gemini Flash on identical inputs.

**Before running:** upload the 13 `.txt` documents into the `docs` folder (cell 2 creates it), and add `ANTHROPIC_API_KEY` and `GEMINI_API_KEY` in Colab Secrets with notebook access switched on.

## 1. Install libraries

The red google-adk message refers to a package Colab preinstalls. It isn’t used here and doesn’t affect the results.

In [ ]:
!pip install -q chromadb sentence-transformers anthropic google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


## 2. Load documents
Each document is under 120 words, so each one is a single chunk.

In [ ]:
import os
os.makedirs("docs", exist_ok=True)

docs = {}
for name in sorted(os.listdir("docs")):
    with open(os.path.join("docs", name), encoding="utf-8") as f:
        docs[name] = f.read()

print(len(docs), "documents loaded")
for name, text in docs.items():
    print(name, len(text.split()), "words")

13 documents loaded
01_duty-log_2023-12-14_trail-wind.txt 64 words
02_email_2023-12-13_lumen-wind-threshold.txt 85 words
03_followup_2023-12-16_trail-wind-outcome.txt 69 words
04_duty-log_2026-07-04_nocturne-wind.txt 46 words
05_guest-recovery_2019-05-23_gardens-maintenance.txt 63 words
06_guest-recovery_2022-08-11_south-lawn-flood.txt 114 words
07_email_2024-02-06_northgate-access.txt 100 words
08_note_2024-02-06_filming-access-resolution.txt 117 words
09_duty-log_2024-03-08_routine.txt 41 words
10_email_2023-09-19_lost-property.txt 45 words
11_handover_2025-06-02.txt 40 words
12_duty-log_2024-10-27_half-term.txt 39 words
13_email_2026-01-14_annual-pass-renewals.txt 32 words


## 3. Embed and store in Chroma (two embedding models)

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

client = chromadb.Client()
names = list(docs.keys())
texts = list(docs.values())

def build(model_name):
    m = SentenceTransformer(model_name)
    col = client.get_or_create_collection(name=model_name)
    col.upsert(ids=names, documents=texts, embeddings=m.encode(texts).tolist())
    return m, col

models = {}
for model_name in ["all-MiniLM-L6-v2", "all-mpnet-base-v2"]:
    models[model_name] = build(model_name)
    print(model_name, "stored", models[model_name][1].count(), "documents")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (18 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

all-mpnet-base-v2 stored 13 documents


## 4. Test questions and answer key

In [ ]:
questions = [
    "Wind advisory on a sold-out trail night. What did we decide last time?",
    "Gardens have to close for maintenance but visitors already have tickets. What do we do?",
    "A contractor can't get access because an event is blocking the gate. How was this handled before?",
]

expected = {
    questions[0]: {"01_duty-log_2023-12-14_trail-wind.txt",
                   "02_email_2023-12-13_lumen-wind-threshold.txt",
                   "03_followup_2023-12-16_trail-wind-outcome.txt",
                   "04_duty-log_2026-07-04_nocturne-wind.txt"},
    questions[1]: {"05_guest-recovery_2019-05-23_gardens-maintenance.txt",
                   "06_guest-recovery_2022-08-11_south-lawn-flood.txt"},
    questions[2]: {"07_email_2024-02-06_northgate-access.txt",
                   "08_note_2024-02-06_filming-access-resolution.txt"},
}

## 5. Retrieval evaluation: embeddings only
recall@k = expected documents found in the top k. Max at k=3 is 7/8, because the wind question has 4 correct documents.

In [ ]:
def evaluate_with(m, col, k):
    total_expected = sum(len(exp) for exp in expected.values())
    found, top1 = 0, 0
    for q, exp in expected.items():
        res = col.query(query_embeddings=m.encode([q]).tolist(), n_results=k)
        got = res["ids"][0]
        found += len([n for n in got if n in exp])
        top1 += got[0] in exp
    print(f"recall@{k}: {found}/{total_expected} | top result correct: {top1}/{len(expected)}")

for model_name, (m, col) in models.items():
    print("\n" + model_name)
    evaluate_with(m, col, 3)
    evaluate_with(m, col, 5)


all-MiniLM-L6-v2
recall@3: 7/8 | top result correct: 2/3
recall@5: 8/8 | top result correct: 2/3

all-mpnet-base-v2
recall@3: 7/8 | top result correct: 2/3
recall@5: 8/8 | top result correct: 2/3


## 6. Add a reranker
Embeddings shortlist 5 documents; a cross-encoder reads question and document together and reorders them.

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve_rerank(m, col, question, k_retrieve=5, k_final=3):
    res = col.query(query_embeddings=m.encode([question]).tolist(), n_results=k_retrieve)
    candidates = list(zip(res["ids"][0], res["documents"][0]))
    scores = reranker.predict([(question, text) for _, text in candidates])
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [name for (name, _), _ in ranked[:k_final]]

for model_name, (m, col) in models.items():
    total_expected = sum(len(e) for e in expected.values())
    found, top1 = 0, 0
    print("\n" + model_name + " + reranker")
    for q, exp in expected.items():
        got = retrieve_rerank(m, col, q)
        found += len([n for n in got if n in exp])
        top1 += got[0] in exp
        print(f"  top: {got[0][:45]} | {q[:35]}")
    print(f"recall@3: {found}/{total_expected} | top result correct: {top1}/{len(expected)}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


all-MiniLM-L6-v2 + reranker
  top: 03_followup_2023-12-16_trail-wind-outcome.txt | Wind advisory on a sold-out trail n
  top: 05_guest-recovery_2019-05-23_gardens-maintena | Gardens have to close for maintenan
  top: 08_note_2024-02-06_filming-access-resolution. | A contractor can't get access becau
recall@3: 7/8 | top result correct: 3/3

all-mpnet-base-v2 + reranker
  top: 03_followup_2023-12-16_trail-wind-outcome.txt | Wind advisory on a sold-out trail n
  top: 05_guest-recovery_2019-05-23_gardens-maintena | Gardens have to close for maintenan
  top: 08_note_2024-02-06_filming-access-resolution. | A contractor can't get access becau
recall@3: 7/8 | top result correct: 3/3


## 7. Connect to Claude and Gemini
Keys are read from Colab Secrets, never written in the notebook.

In [ ]:
from google.colab import userdata
import anthropic
from google import genai
from google.genai import types, errors

claude = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
gemini = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

## 8. Grounded generation setup
Chosen setup: MiniLM + reranker. All 5 shortlisted documents are passed to the model, so a conflicting record is not dropped.

In [ ]:
m, col = models["all-MiniLM-L6-v2"]

SYSTEM = """You help duty managers at a heritage estate recall past operational decisions.
Answer ONLY from the records provided. Do not use outside knowledge.
Cite the record's file name in square brackets after each fact.
If the records do not contain a relevant precedent, say so plainly. Do not guess.
Keep the answer under 120 words.
If records disagree with each other, say so and state what each record says. Do not silently pick one."""

def build_context(question):
    names = retrieve_rerank(m, col, question, k_retrieve=5, k_final=5)
    return names, "\n\n".join(f"### {n}\n{docs[n]}" for n in names)

test_questions = questions + [
    "Bats have been found roosting in the chapel roof. What did we do last time?"
]

## 9. Compare Claude Haiku and Gemini Flash on identical inputs
`gemini-3.8-flash` was unavailable (503) across retries, so the comparison uses `gemini-3.5-flash`.

In [ ]:
import time

GEMINI_MODEL = "gemini-3.5-flash"

def claude_answer(context, question, model="claude-haiku-5-5"):
    reply = claude.messages.create(
        model=model, max_tokens=2000, system=SYSTEM,
        messages=[{"role": "user", "content": f"Records:\n{context}\n\nQuestion: {question}"}],
    )
    text = "".join(b.text for b in reply.content if b.type == "text")
    return text, reply.usage.input_tokens, reply.usage.output_tokens

def gemini_answer(context, question, model=GEMINI_MODEL, attempts=4):
    for attempt in range(attempts):
        try:
            resp = gemini.models.generate_content(
                model=model,
                contents=f"Records:\n{context}\n\nQuestion: {question}",
                config=types.GenerateContentConfig(system_instruction=SYSTEM, max_output_tokens=2000),
            )
            u = resp.usage_metadata
            out = (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)
            return resp.text, u.prompt_token_count, out
        except errors.ServerError:
            wait = 15 * (attempt + 1)
            print(f"  Gemini busy (attempt {attempt + 1} of {attempts}), waiting {wait}s")
            time.sleep(wait)
    return "[NO ANSWER: model unavailable after retries]", 0, 0

results = []
for q in test_questions:
    names, context = build_context(q)
    for label, fn in [("Claude Haiku", claude_answer), ("Gemini Flash", gemini_answer)]:
        text, tin, tout = fn(context, q)
        results.append({"question": q, "model": label, "answer": text, "in": tin, "out": tout})
        print("\n" + "=" * 60)
        print(f"[{label}] Q: {q}")
        print(text)
        print(f"tokens in/out: {tin}/{tout}")


[Claude Haiku] Q: Wind advisory on a sold-out trail night. What did we decide last time?
The records contain a relevant precedent from 14 December 2023, but they disagree on what was done:

- **Duty log (16:40):** Closed the East Gate section of the Trail from 18:00 and kept the rest running. About 400 guests were redirected via the Water Terraces [01].
- **Follow-up note (16 Dec):** The path was closed at 15:00, before Lumen's call, and guests were cleared from the trail. Cancellations were refunded [03].

Both records say Lumen's arches come down at 40mph sustained [01][02]. The records don't confirm which closure version happened.

A July 2026 Nocturne case involved a 30mph forecast; the Head of Events chose not to act and the event ran in full [04]. That was not a trail night.
tokens in/out: 1212/1019
  Gemini busy (attempt 1 of 4), waiting 15s
  Gemini busy (attempt 2 of 4), waiting 30s
  Gemini busy (attempt 3 of 4), waiting 45s

[Gemini Flash] Q: Wind advisory on a sold-out tra

## 10. Save results to Google Drive

In [ ]:
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
pd.DataFrame(results).to_csv("/content/drive/MyDrive/rag_model_comparison.csv", index=False)
print(len(results), "answers saved")

Mounted at /content/drive
8 answers saved


In [2]:
test_questions = [
    "Wind advisory on a sold-out trail night. What did we decide last time?",
    "Gardens have to close for maintenance but visitors already have tickets. What do we do?",
    "A contractor can't get access because an event is blocking the gate. How was this handled before?",
    "Bats have been found roosting in the chapel roof. What did we do last time?",
]
answer_key = {
    test_questions[0]: {
        "must_include": [
            "Records conflict on 14 Dec 2023: duty log says only the East Gate section closed from 18:00 and the rest kept running; follow-up says the trail was closed and guests cleared",
            "The decision to close was taken at 15:00, ahead of Lumen's call",
            "Lumen's arches come down at sustained winds over 40mph",
            "Lumen asked for a decision by 16:00; take-down needs a crew of four and about 90 minutes",
            "Nocturne, 4 Jul 2026: 30mph forecast, contractor liability at 20mph, sensors never above 15mph, Head of Events decided not to act, event ran in full",
        ],
        "must_not_say": [
            "That the trail closed at 15:00 (15:00 is when the decision was taken)",
            "That cancellations were refunded (the website said they would be)",
            "Any reason for the Nocturne decision (the log records the decision, not why)",
            "That either event was sold out",
            "That one 2023 account is the correct one",
            "Any fact without a citation",
        ],
    },
    test_questions[1]: {
        "must_include": [
            "2019 announced maintenance closure: guests who arrived unaware were offered a shop or cafe voucher, or tickets converted to annual passes",
            "2022 unannounced closure: partial refunds at the gate landed poorly and varied between staff",
            "Agreed position for unannounced closures of a ticketed area: full refund or return pass",
        ],
        "must_not_say": [
            "That no record covers a planned or announced closure",
            "That the full refund or return pass rule applies to announced closures",
            "Any fact without a citation",
        ],
    },
    test_questions[2]: {
        "must_include": [
            "Engineer escorted in via the courtyard entrance between takes",
            "About three hours lost; job ran to 17:00 instead of 13:00",
            "The estate paid the full day rate",
            "Contractor bookings and filming/event schedules are not cross-checked",
        ],
        "must_not_say": [
            "That the contractor was charged or paid",
            "That the engineer was turned away",
            "Any fact without a citation",
        ],
    },
    test_questions[3]: {
        "must_include": [
            "The records contain no precedent for bats or the chapel",
        ],
        "must_not_say": [
            "Any invented precedent",
            "Advice or recommendations from outside the records",
            "A wrong count of the records provided",
            "Unrelated records presented as guidance",
        ],
    },
}
print(len(answer_key), "questions in answer key")

4 questions in answer key
